In [14]:
import json
import re
import random
import os
from datasets import load_dataset
from tqdm import tqdm

from dotenv import load_dotenv

In [15]:
load_dotenv()

True

In [16]:
DATASET_NAME = "glaiveai/glaive-function-calling-v2"
OUTPUT_DIR = "data"

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

MIN_EXAMPLES = 2000
TARGET_TOTAL = 5000
RANDOM_SEED = 42

In [17]:
INSTRUCTIONS = [
    "Extract the key information from the following text and return it as JSON.",
    "Parse the input text and output structured JSON with entities and values.",
    "Extract all explicit entities and values as JSON.",
    "Convert the text into structured JSON format.",
    "Identify and extract relevant structured information from the input text.",
    "Return only JSON containing extracted fields."
]

In [18]:
def extract_user_message(chat_text):
    pattern = r'USER:\s*(.*?)(?:\n\n|ASSISTANT:|$)'
    match = re.search(pattern, chat_text, re.DOTALL)
    return match.group(1).strip() if match else None

In [19]:
def extract_functioncall_json(chat_text):
    pattern = r'<functioncall>\s*({.*?})(?:\s*<\|endoftext\|>|$)'
    matches = re.findall(pattern, chat_text, re.DOTALL)

    results = []
    for m in matches:
        try:
            results.append(json.loads(m))
        except:
            pass
    return results

In [20]:
def clean_extraction_json(parsed_json):
    if not isinstance(parsed_json, dict):
        return {}

    result = {}

    if "arguments" in parsed_json:
        result.update(parsed_json["arguments"])

    for k, v in parsed_json.items():
        if k not in ["name", "arguments"]:
            result[k] = v

    return result

In [21]:
def validate_example(ex):
    if not ex:
        return False
    if not ex["input"] or len(ex["input"]) < 5:
        return False

    try:
        json.loads(ex["output"])
    except:
        return False

    return True

In [22]:
def format_for_training(instruction, user_input, extracted_json):
    return {
        "instruction": instruction,
        "input": user_input,
        "output": json.dumps(extracted_json, ensure_ascii=False),
        "text": f"""### Instruction:
{instruction}

### Input:
{user_input}

### Response:
{json.dumps(extracted_json, ensure_ascii=False)}"""
    }

In [23]:
def convert_example(row):
    chat = row.get("chat", "")

    user_msg = extract_user_message(chat)
    if not user_msg:
        return None

    fc_jsons = extract_functioncall_json(chat)
    if not fc_jsons:
        return None

    cleaned = clean_extraction_json(fc_jsons[0])

    instruction = random.choice(INSTRUCTIONS)

    return format_for_training(instruction, user_msg, cleaned)

In [24]:
print("Loading dataset...")

ds = load_dataset(DATASET_NAME, split="train")

print("Loaded:", len(ds))

Loading dataset...
Loaded: 112960


In [25]:
print(ds)
print(type(ds))

Dataset({
    features: ['system', 'chat'],
    num_rows: 112960
})
<class 'datasets.arrow_dataset.Dataset'>


In [26]:
converted = []
skipped = 0

data_list = list(ds)

for row in tqdm(data_list):
    ex = convert_example(row)

    if validate_example(ex):
        converted.append(ex)
    else:
        skipped += 1

    if len(converted) >= TARGET_TOTAL:
        break

print("Valid:", len(converted))
print("Skipped:", skipped)

 35%|███▍      | 39034/112960 [00:01<00:02, 30985.82it/s]


TypeError: 'NoneType' object is not iterable

In [ ]:
random.seed(RANDOM_SEED)
random.shuffle(converted)

n = len(converted)

n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)

train = converted[:n_train]
val = converted[n_train:n_train+n_val]
test = converted[n_train+n_val:]

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save(name, data):
    path = os.path.join(OUTPUT_DIR, f"{name}.jsonl")
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"{name}: {len(data)} saved")

save("train", train)
save("val", val)
save("test", test)

In [ ]:
print("\nDONE ✅")
print("Train:", len(train))
print("Val:", len(val))
print("Test:", len(test))
print("Total:", len(converted))

In [ ]:
# import json

# with open("data/processed/train.jsonl", "w") as f:
#     for item in processed:
#         f.write(json.dumps(item) + "\n")